In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic


In [3]:

!pip install -q timm thop scikit-learn xgboost torchinfo

import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import timm
from thop import profile
from torchinfo import summary
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# Cell 2: Load Real ISIC Dataset from kagglehub Path
import os
import kagglehub
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms

# Download dataset
path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

# Define standard image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Determine the correct root paths for training and testing data
# Based on typical Kaggle dataset structure: path -> main_folder -> (Train, Test) -> classes
base_data_path = os.path.join(path, 'Skin cancer ISIC The International Skin Imaging Collaboration')
train_data_path = os.path.join(base_data_path, 'Train')
test_data_path = os.path.join(base_data_path, 'Test')

# Load training and testing datasets separately
train_dataset = ImageFolder(root=train_data_path, transform=transform)
test_dataset = ImageFolder(root=test_data_path, transform=transform)

print(f"Total training images found: {len(train_dataset)}")
print(f"Total testing images found: {len(test_dataset)}")
print(f"Classes for training: {train_dataset.classes}")
print(f"Classes for testing: {test_dataset.classes}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Total training images found: 2239
Total testing images found: 118
Classes for training: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Classes for testing: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']


In [5]:
# Cell 3 (Corrected): Multi-Epoch Transfer Learning Training & Evaluation Loop (Tables 1 & 3)
model_names = ['vgg16', 'resnet18', 'efficientnet_b0']
results_table1 = []
results_table3 = []
num_epochs = 20  # Train for multiple epochs instead of 1 batch

for name in model_names:
    print(f"\n--- Training & Benchmarking: {name} ---")
    model = timm.create_model(name, pretrained=True, num_classes=9).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    # 1. Compute Efficiency Metrics (Table 3 inputs)
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    try:
        macs, params = profile(model, inputs=(dummy_input,), verbose=False)
        flops_g = (macs * 2) / 1e9
    except:
        flops_g, params = 0.0, sum(p.numel() for p in model.parameters())

    param_m = params / 1e6

    # Model Size in MB
    torch.save(model.state_dict(), "temp.pth")
    model_size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # 2. Proper Multi-Epoch Training Loop (Removed the 'break' statement)
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

    # 3. Evaluation & Inference Time (Table 1 & 3)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    start_time = time.time()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    total_time = time.time() - start_time
    inference_time_ms = (total_time / len(test_dataset)) * 1000

    acc = accuracy_score(all_labels, all_preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

    # Fixed AUC Calculation
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro') * 100
    except Exception as e:
        print(f"AUC calculation warning: {e}")
        auc = 0.0

    results_table1.append({
        "Model": name, "Accuracy (%)": f"{acc:.2f}", "Precision (%)": f"{precision*100:.2f}",
        "Recall (%)": f"{recall*100:.2f}", "F1-Score (%)": f"{f1*100:.2f}", "AUC (%)": f"{auc:.2f}"
    })

    results_table3.append({
        "Model": name, "Parameters (M)": f"{param_m:.2f}", "Model Size (MB)": f"{model_size_mb:.2f}",
        "FLOPs (G)": f"{flops_g:.2f}", "Inference Time (ms)": f"{inference_time_ms:.2f}", "Accuracy (%)": f"{acc:.2f}"
    })

print("\nPipeline Complete!")


--- Training & Benchmarking: vgg16 ---


model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [1/20], Loss: 1.4709
Epoch [2/20], Loss: 0.9749
Epoch [3/20], Loss: 0.6587
Epoch [4/20], Loss: 0.4326
Epoch [5/20], Loss: 0.3658
Epoch [6/20], Loss: 0.3863
Epoch [7/20], Loss: 0.2677
Epoch [8/20], Loss: 0.2068
Epoch [9/20], Loss: 0.1907
Epoch [10/20], Loss: 0.1820
Epoch [11/20], Loss: 0.1895
Epoch [12/20], Loss: 0.1672
Epoch [13/20], Loss: 0.3119
Epoch [14/20], Loss: 0.2363
Epoch [15/20], Loss: 0.1559
Epoch [16/20], Loss: 0.1763
Epoch [17/20], Loss: 0.1571
Epoch [18/20], Loss: 0.1684
Epoch [19/20], Loss: 0.1414
Epoch [20/20], Loss: 0.1293

--- Training & Benchmarking: resnet18 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [1/20], Loss: 2.0305
Epoch [2/20], Loss: 1.7544
Epoch [3/20], Loss: 1.5089
Epoch [4/20], Loss: 1.2782
Epoch [5/20], Loss: 1.0942
Epoch [6/20], Loss: 0.9547
Epoch [7/20], Loss: 0.8406
Epoch [8/20], Loss: 0.7364
Epoch [9/20], Loss: 0.6579
Epoch [10/20], Loss: 0.5772
Epoch [11/20], Loss: 0.5027
Epoch [12/20], Loss: 0.4419
Epoch [13/20], Loss: 0.3943
Epoch [14/20], Loss: 0.3381
Epoch [15/20], Loss: 0.2999
Epoch [16/20], Loss: 0.2683
Epoch [17/20], Loss: 0.2447
Epoch [18/20], Loss: 0.2222
Epoch [19/20], Loss: 0.2108
Epoch [20/20], Loss: 0.1833

--- Training & Benchmarking: efficientnet_b0 ---


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch [1/20], Loss: 2.2992
Epoch [2/20], Loss: 0.7604
Epoch [3/20], Loss: 0.4205
Epoch [4/20], Loss: 0.2920
Epoch [5/20], Loss: 0.2657
Epoch [6/20], Loss: 0.2405
Epoch [7/20], Loss: 0.2223
Epoch [8/20], Loss: 0.1980
Epoch [9/20], Loss: 0.2057
Epoch [10/20], Loss: 0.1918
Epoch [11/20], Loss: 0.1869
Epoch [12/20], Loss: 0.1726
Epoch [13/20], Loss: 0.1566
Epoch [14/20], Loss: 0.1536
Epoch [15/20], Loss: 0.1533
Epoch [16/20], Loss: 0.1405
Epoch [17/20], Loss: 0.1402
Epoch [18/20], Loss: 0.1446
Epoch [19/20], Loss: 0.1384
Epoch [20/20], Loss: 0.1444

Pipeline Complete!


In [6]:
# Cell 4: Deep Feature Extraction & Classical Classifiers (Table 2)
# Using a feature extractor backbone (e.g., ResNet18 without final classification head)
backbone = timm.create_model('resnet18', pretrained=True, num_classes=0).to(device)
backbone.eval()

def extract_features(loader):
    features, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            feats = backbone(images)
            features.append(feats.cpu().numpy())
            targets.append(labels.numpy())
    return np.vstack(features), np.concatenate(targets)

X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Linear SVM": LinearSVC(max_iter=1000),
    "RBF-SVM": SVC(probability=True),
    "XGBoost": XGBClassifier()
}

table2_results = []
for clf_name, clf in classifiers.items():
    print(f"Training Classifier: {clf_name}...")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)

    acc = accuracy_score(y_test, preds) * 100
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average='macro', zero_division=0)

    table2_results.append({
        "Classifier": clf_name, "Accuracy (%)": f"{acc:.2f}",
        "Precision (%)": f"{prec*100:.2f}", "Recall (%)": f"{rec*100:.2f}",
        "F1-Score (%)": f"{f1*100:.2f}"
    })

Training Classifier: Logistic Regression...
Training Classifier: Decision Tree...
Training Classifier: Random Forest...
Training Classifier: K-Nearest Neighbors (KNN)...
Training Classifier: Linear SVM...
Training Classifier: RBF-SVM...
Training Classifier: XGBoost...


In [7]:
import pandas as pd

In [8]:
import pandas as pd
df_table1 = pd.DataFrame(results_table1)
display(df_table1)

,Model,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,vgg16,49.15,47.49,49.31,44.70,87.02
1,resnet18,55.08,60.16,54.17,48.21,90.92
2,efficientnet_b0,52.54,51.37,52.08,48.14,87.77


In [9]:
df_table2 = pd.DataFrame(table2_results)
display(df_table2)

,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%)
0,Logistic Regression,50.00,47.74,50.00,44.43
1,Decision Tree,23.73,18.35,25.46,19.81
2,Random Forest,36.44,37.46,38.89,28.50
3,K-Nearest Neighbors (KNN),28.81,32.71,29.63,27.93
4,Linear SVM,45.76,45.09,46.53,42.63
5,RBF-SVM,49.15,52.82,49.31,44.43
6,XGBoost,42.37,40.18,43.75,37.99


In [10]:
df_table3 = pd.DataFrame(results_table3)
display(df_table3)

,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,vgg16,134.30,512.32,30.93,67.51,49.15
1,resnet18,11.18,42.74,3.65,62.37,55.08
2,efficientnet_b0,3.98,15.73,0.77,62.47,52.54
